In [ ]:
# PED: análisis básico de datos climáticos con ERA5

En esta práctica se trabajará con datos del reanálisis ERA5 descargados por el propio estudiante desde el portal Copernicus Climate Data Store (CDS).

El objetivo es doble:

1. representar una serie temporal diaria de precipitación para un punto próximo a Madrid;
2. elaborar un mapa de precipitación sobre la Comunidad de Madrid a partir de datos en rejilla.

La práctica permite familiarizarse con un flujo de trabajo muy habitual en análisis climático:

- localización y descarga de datos;
- carga del archivo en un entorno de análisis;
- inspección de variables y dimensiones;
- transformación básica de unidades y escalas temporales;
- representación gráfica e interpretación.

## Recordatorio importante

En ERA5, la precipitación total (`total precipitation`, `tp`) se expresa en **metros de agua**. Para expresarla en milímetros, se multiplica por 1000. :contentReference[oaicite:2]{index=2}

In [ ]:
## Descarga previa de los datos

Antes de ejecutar este notebook, descargue desde Copernicus los archivos necesarios.

### Archivo 1. Serie temporal puntual
Dataset recomendado:
**ERA5 hourly time-series data on single levels from 1940 to present**. :contentReference[oaicite:3]{index=3}

Configuración orientativa:
- Variable: `total precipitation`
- Localización: Madrid (por ejemplo, latitud 40.42; longitud -3.70)
- Periodo: junio-agosto de 2019
- Formato: CSV

### Archivo 2. Campo espacial para mapa
Dataset recomendado:
**ERA5 hourly data on single levels from 1940 to present**. :contentReference[oaicite:4]{index=4}

Configuración orientativa:
- Variable: `total precipitation`
- Área: entorno de la Comunidad de Madrid
- Fecha: elija un día del verano de 2019
- Hora: puede descargar todas las horas del día si quiere construir precipitación diaria
- Formato: NetCDF

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
## 2. Serie temporal de precipitación en un punto

En este primer apartado trabajaremos con el archivo CSV descargado para un punto próximo a Madrid.

El objetivo es obtener una **serie diaria de precipitación** a partir de datos horarios.

### Tarea
Escriba en la variable `csv_file` el nombre exacto de su archivo descargado.

In [ ]:
csv_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.csv"

df = pd.read_csv(csv_file)
df.head()

In [ ]:
### Pregunta 1
Observe la tabla cargada y responda brevemente:

- ¿qué columnas aparecen?
- ¿qué columna contiene la fecha y hora?
- ¿qué columna contiene la precipitación?

In [ ]:
df.columns

In [ ]:
## 3. Conversión de la fecha y hora

Convierta la columna temporal al tipo fecha-hora de `pandas`.

Después, cree una nueva columna que contenga solo la fecha, sin la hora.

In [ ]:
# Sustituya el nombre de la columna temporal si fuera necesario
df["valid_time"] = pd.to_datetime(df["valid_time"])

# Cree una columna solo con la fecha
df["date"] = df["valid_time"].dt.date

df.head()

In [ ]:
## 4. Unidades de precipitación

En ERA5, la variable `total precipitation` (`tp`) se expresa en metros de agua. Para pasar a milímetros, se multiplica por 1000. :contentReference[oaicite:5]{index=5}

### Tarea
Cree una nueva columna con la precipitación en milímetros.

In [ ]:
# Sustituya 'tp' por el nombre real de la columna si fuera distinto
df["tp_mm"] = df["tp"] * 1000

df[["valid_time", "tp", "tp_mm"]].head()

In [ ]:
## 5. De datos horarios a precipitación diaria

A diferencia de la temperatura media, la precipitación diaria no se obtiene promediando los valores horarios, sino **sumándolos**.

### Tarea
Agrupe los datos por día y calcule la precipitación total diaria.

In [ ]:
daily_precip = (
    df.groupby("date", as_index=False)["tp_mm"]
      .sum()
      .rename(columns={"tp_mm": "precip_mm_day"})
)

daily_precip.head()

In [ ]:
## 6. Representación de la serie temporal

Represente la serie diaria obtenida.

### Pregunta 2
Describa brevemente el comportamiento de la precipitación en el periodo analizado.
No se trata solo de decir si llueve más o menos, sino de indicar si la precipitación aparece de forma continua o concentrada en unos pocos episodios.

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(daily_precip["date"], daily_precip["precip_mm_day"])
plt.xlabel("Fecha")
plt.ylabel("Precipitación diaria (mm)")
plt.title("Precipitación diaria en Madrid (ERA5)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
## 7. Mapa de precipitación sobre la Comunidad de Madrid

En este segundo apartado trabajaremos con un archivo NetCDF descargado desde ERA5 en rejilla.

El objetivo es generar un mapa de precipitación para la región de Madrid.

In [ ]:
nc_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.nc"

ds = xr.open_dataset(nc_file)
ds

In [ ]:
### Pregunta 3
A partir de la salida del dataset, indique:

- nombre de la variable principal de precipitación;
- dimensiones disponibles;
- coordenadas espaciales incluidas.